## Personally identifiable information (PII) in the dataset

###  Definition of PII

Personally Identifiable Information (PII) refers to any information that can identify a specific individual, either directly or indirectly.

Under the General Data Protection Regulation (GDPR), the broader term used is “personal data”, defined as "Any information relating to an identified or identifiable natural person."

An identifiable person is someone who can be identified directly (e.g., by name or ID number) or indirectly (e.g., through a combination of attributes such as date of birth, gender, and ZIP code).

Since the NovaCred dataset is used for credit scoring decisions, it contains multiple forms of personal data that fall under GDPR protection and EU AI Act obligations.


###  PII classification
We have audited the `raw_credit_applications.json` dataset and identified several categories of Personally Identifiable Information (PII). Under GDPR, these require specific technical and organizational measures to ensure lawful processing.

#### 1 Direct Identifiers
These fields can be used to uniquely identify a natural person without any additional information:
* `full_name`: Directly identifies a person. It's the most straightforward form of PII since it points to a specific individual without needing any additional context.
* `email`: Uniquely assigned to an individual, often contains the person's name, and serves as both an identifier and a contact channel. Even without a name attached, an email address can be traced back to its owner.
* `ssn`: A permanent, unique national identifier assigned to a single person. It's among the most sensitive PII because it cannot be easily changed and is widely used for identity verification across financial, governmental, and healthcare systems. Exposure creates severe risk of identity theft.
* `ip_adress`: Identifies the device and network used during the application. It can be linked back to a specific individual through their internet service provider, which means it qualifies as personal data even if it doesn't directly contain a name.

---




#### 2 Indirect identifiers (quasi-identifiers)

Not identifying on their own, but become PII when combined with other fields:

* `date_of_birth`:  A birthdate alone is shared by many people. However, when combined with other quasi-identifiers like gender and zip code, the pool of matching individuals shrinks dramatically, often to a single person. The risk lies in combination, not isolation.

* `gender`: Same principle. Knowing someone's gender tells you very little by itself, but it significantly narrows the population when crossed with location and age data. It's also a protected characteristic under anti-discrimination legislation, adding a layer of sensitivity beyond identification.

* `zip_code`: Geographic information that narrows someone's location to a small area. Some postal codes cover very few residents, which makes re-identification much easier. It also carries indirect sensitivity because residential patterns often correlate with ethnicity and socioeconomic status.

---

#### 3 Behavioral Data with Privacy Implications

* `spending_behavior` (category + amount): Not a traditional identifier, but highly personal. Detailed spending patterns can reveal sensitive aspects of someone's life such as health conditions, religious beliefs, or political affiliations depending on the categories tracked. This raises a data minimization concern: is this level of granularity necessary for the purpose of credit decisioning?
---
#### 4 Non-PII fields

* `_id`: An internal application identifier with no meaning outside the system. It doesn't relate to the person's real-world identity.

* `annual_income`, `credit_history_months`, `debt_to_income`, `savings_balance`: These are financial attributes, not identifiers. They describe characteristics rather than pointing to a specific person. However, they become personal data the moment they are linked to any of the identifiers above, because they then describe an identified individual's financial situation.

* decision object (`loan_approved`, `interest_rate`, `approved_amount`, `rejection_reason`): Not PII on its own, but once linked to an applicant it represents the outcome of automated profiling. This makes it relevant under GDPR provisions on automated decision-making, since the decision directly affects the individual's access to credit.

## Implementing Pseudonymization

### SSN Pseudonymization Demo

The SSN (Social Security Number) field is the most sensitive direct identifier in the dataset. It is a permanent, unique national identifier that cannot be changed, making its exposure a severe identity theft risk.

To protect this field while preserving analytical utility, we apply **SHA-256 hashing**, a one-way cryptographic function that converts each SSN into a fixed-length string of characters.

**Why hashing works for this use case:**
- **Irreversible:** the original SSN cannot be recovered from the hash
- **Deterministic:** the same SSN always produces the same hash, so we can still detect duplicate records without seeing the real values
- **Fixed output:** regardless of input, the hash is always 64 characters, removing any structural clues about the original

**GDPR note:** hashed data is still considered *pseudonymized* (not *anonymized*) under GDPR Article 4(5), because re-identification remains theoretically possible. This means GDPR still applies to the hashed dataset.

In [2]:
import pandas as pd
import hashlib

df = pd.read_csv("cleaned_credit_applications.csv")

# Show original SSNs
print("=== Before Pseudonymization ===")
print(df[["_id", "full_name", "ssn"]].head())

# Apply SHA-256 hashing
def hash_ssn(ssn):
    return hashlib.sha256(ssn.encode()).hexdigest()

df["ssn_hashed"] = df["ssn"].apply(hash_ssn)

# Show the result
print("\n=== After Pseudonymization ===")
print(df[["_id", "full_name", "ssn", "ssn_hashed"]].head())

# Drop the original SSN from the working dataset
df = df.drop(columns=["ssn"])

print("\n=== Final: Original SSN removed from dataset ===")
print(df[["_id", "full_name", "ssn_hashed"]].head())

=== Before Pseudonymization ===
       _id        full_name          ssn
0  app_200      Jerry Smith  596-64-4340
1  app_037   Brandon Walker  425-69-4784
2  app_215      Scott Moore  370-78-5178
3  app_024       Thomas Lee  194-35-1833
4  app_184  Brian Rodriguez  480-41-2475

=== After Pseudonymization ===
       _id        full_name          ssn  \
0  app_200      Jerry Smith  596-64-4340   
1  app_037   Brandon Walker  425-69-4784   
2  app_215      Scott Moore  370-78-5178   
3  app_024       Thomas Lee  194-35-1833   
4  app_184  Brian Rodriguez  480-41-2475   

                                          ssn_hashed  
0  2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...  
1  2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...  
2  db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...  
3  c835719be02018987096d6e49529a24b1d7e7ab35c84b1...  
4  41c7de40dc49185886e6ecb37346ec9eabce16087b7508...  

=== Final: Original SSN removed from dataset ===
       _id        full_name                  

## GDPR Mapping
### The 7 Principles (Article 5)

#### 1. Lawfulness, Fairness and Transparency — Art. 5(1)(a)

**Lawfulness:** The dataset contains no record of the legal basis under which personal data was collected and processed. Under GDPR Article 6, every processing activity must rely on one of six lawful bases (e.g., consent, contractual necessity, legitimate interest). NovaCred cannot currently demonstrate which basis applies to each applicant's data.

**Fairness:** The bias analysis reveals disparate impact in loan approval rates by gender and age. Female applicants are approved at ~50.6% compared to ~62.9% for males, producing a Disparate Impact ratio of ~0.80, below the 0.8 threshold commonly used as a fairness benchmark. The interaction analysis further shows that women aged 26–35 are approved at just ~34.5% compared to ~51.2% for men in the same age group, and women aged 56–65 at ~52.4% compared to ~77.8% for men. Decisions that systematically disadvantage protected groups undermine the fairness requirement, especially in automated decision-making contexts.

**Transparency:** Applicants who are rejected receive the reason `algorithm_risk_score`, which provides no meaningful explanation of how the decision was reached. Under GDPR, individuals have the right to understand how their data is being used and how decisions about them are made.

---

#### 2. Purpose Limitation — Art. 5(1)(b)

Personal data must be collected for specified, explicit and legitimate purposes. The dataset contains no documentation of the stated purpose for collecting each field. For instance:
- Is the `ip_address` collected for fraud detection, geolocation, or security logging? Without a declared purpose, its collection cannot be justified.
- Granular `spending_behavior` categories (e.g., Alcohol, Healthcare, Fitness) go beyond what is needed for a credit assessment. If the stated purpose is creditworthiness evaluation, collecting categories that reveal health habits or lifestyle choices exceeds that purpose.

---

#### 3. Data Minimization — Art. 5(1)(c)

Only data that is adequate, relevant and necessary for the stated purpose should be collected. Several fields raise concerns:
- **`ip_address`:** Not required for credit decisioning. A credit application does not need to know the applicant's network address.
- **`spending_behavior` categories:** Aggregate spending totals would suffice for assessing financial behavior. Granular categories like Healthcare or Alcohol can reveal sensitive information about health conditions or personal habits that is not necessary for evaluating creditworthiness.
- **`ssn`:** While needed for identity verification, it should not be stored in plain text after verification is complete. The principle requires minimizing not just what is collected, but how it is retained.

---

#### 4. Accuracy — Art. 5(1)(d)

Data must be accurate and kept up to date. The data quality analysis identified several accuracy issues in the dataset:
- **Inconsistent gender encoding:** The `gender` column contained four different representations: `Male` (195), `Female` (193), `M` (53), `F` (58), plus 2 empty strings and 1 NaN, meaning the same attribute was recorded inconsistently across records.
- **Invalid numerical values:** 2 records with negative `credit_history_months`, 1 record with negative `savings_balance`, 1 record with `debt_to_income` greater than 1.0, and 1 record with `annual_income` equal to 0, values that do not reflect reality.
- **Missing data in key fields:** 5 records missing `annual_income`, 1 missing `gender`, and highly sparse columns such as `processing_timestamp` (87.6% missing) and `loan_purpose` (90% missing).

Each of these issues means that automated credit decisions are being made on inaccurate or incomplete data, directly violating this principle.

---

#### 5. Storage Limitation — Art. 5(1)(e)

Personal data must be kept only for as long as necessary for the purpose it was collected. The dataset has:
- **No retention policy:** There is no field indicating when records should be deleted or archived (e.g., `retention_until`, `created_at`, `expires_at`).
- **Effectively no timestamps:** A `processing_timestamp` field exists but is missing for 87.6% of records (440 out of 502), making it useless for determining how long data has been stored.
- **No deletion mechanism:** There is no evidence of a process for periodically reviewing and purging records that are no longer needed.

---

#### 6. Integrity and Confidentiality — Art. 5(1)(f)

Data must be protected against unauthorized access, loss or damage. The dataset stores highly sensitive identifiers in plain text with no protection:
- **`ssn`:** Social Security Numbers are fully visible. A single data breach would expose permanent, irreplaceable national identifiers for every applicant in the dataset.
- **`full_name`, `email`:** Stored without any encryption or pseudonymization.
- **`ip_address`:** Stored in plain text, linkable to individuals through ISP records.
- **No access controls documented:** There is no indication of who can access this data, under what conditions, or with what level of authorization.

---

#### 7. Accountability — Art. 5(2)

The data controller must be able to demonstrate compliance with all of the above principles. NovaCred currently lacks:
- **No audit trail:** There is no log of who accessed the data, when decisions were made, or what factors influenced each outcome.
- **No consent records:** Without a `consent_timestamp` or `consent_type` field, NovaCred cannot prove that applicants agreed to the processing of their data.
- **No documentation of processing activities:** GDPR Article 30 requires a Record of Processing Activities (ROPA). The dataset shows no evidence that this exists.
- **No Data Protection Impact Assessment (DPIA):** Given that the system involves automated decision-making with legal effects on individuals (credit decisions), a DPIA is mandatory under Article 35. There is no indication one has been conducted.

## EU AI Act Classification

### NovaCred as a High-Risk AI System

Under the EU AI Act's risk-based framework, AI systems used for **credit scoring and creditworthiness assessment** are explicitly classified as **High Risk** (Annex III, Section 5(b)). This means NovaCred's lending model is subject to the strictest set of requirements before it can be deployed or continue operating.

### Requirements and Findings

**Risk Management System — Art. 9**

High-risk AI systems must have a continuous risk management process that identifies and mitigates risks throughout the system lifecycle. The dataset shows no evidence of:
- A documented risk assessment for the lending model
- Identification of known risks (bias, data quality issues, lack of explainability)
- Mitigation measures in place for any of these risks
- Monitoring processes to detect when the model's behavior drifts or degrades

The data quality issues (inconsistent encoding, invalid values, missing data) and the bias patterns found in the analysis are exactly the kind of risks this system should have identified and addressed before deployment. Specifically:
- **Gender bias:** Disparate Impact ratio of ~0.80, with female applicants approved at ~50.6% versus ~62.9% for males.
- **Intersectional bias:** Women aged 26–35 approved at ~34.5% versus ~51.2% for men; women aged 56–65 at ~52.4% versus ~77.8% for men.
- **Proxy discrimination:** `zip_code` shows a -0.82 correlation with `gender_binary`, meaning zip code could act as a proxy for gender even if gender were removed from the model. This is a risk that should have been identified and mitigated.

---

**Data Quality and Governance — Art. 10**

Training, validation and testing datasets for high-risk systems must be relevant, representative, free of errors, and complete. The NovaCred dataset fails on multiple counts:
- **Not free of errors:** Inconsistent gender coding (4 different representations: `Male`, `Female`, `M`, `F`), 2 records with negative `credit_history_months`, 1 with negative `savings_balance`, 1 with `debt_to_income` above 1.0, and 1 with `annual_income` equal to 0.
- **Not representative:** The Disparate Impact ratio of ~0.80 indicates that the model systematically disadvantages female applicants. The interaction analysis shows this disparity is even more pronounced in specific age groups, suggesting the training data or the model reflects discriminatory patterns.
- **Completeness concerns:** 5 records missing `annual_income`, 1 missing `gender`, and highly sparse columns including `processing_timestamp` (87.6% missing), `loan_purpose` (90% missing), and `notes` (99.6% missing), meaning the model is making decisions on incomplete information.
- **No data governance:** There is no documentation of how the dataset was collected, curated, or validated before being used for automated decisions.

---

**Transparency and Information — Art. 13**

High-risk systems must be designed to allow users to interpret the system's output. NovaCred fails this requirement:
- **`algorithm_risk_score`** provides no interpretable information about why a decision was made, it is the only rejection reason given to denied applicants.
- There is no feature importance or factor breakdown available for any decision.
- Applicants and internal reviewers alike cannot understand what drives the model's outcomes.
- No technical documentation exists describing the model's capabilities, limitations, or known biases.

---

**Human Oversight — Art. 14**

High-risk systems must be designed to allow effective human oversight, including the ability to understand the system's outputs, override decisions, and intervene when necessary. The dataset shows:
- **No human review fields:** No `reviewed_by`, `override_decision`, or `human_approval` column exists.
- **No intervention mechanism:** There is no process for a human to review and override automated rejections.
- **Full automation:** The combination of `algorithm_risk_score` rejections and the absence of any human oversight fields indicates that credit decisions are fully automated with no human in the loop.

This is particularly concerning because the EU AI Act requires that high-risk systems allow humans to override or reverse automated decisions, especially when those decisions have significant effects on individuals.

---

**Record-Keeping — Art. 12**

High-risk systems must automatically log events to ensure traceability. The dataset contains:
- **Effectively no timestamps:** A `processing_timestamp` field exists but is missing for 87.6% of records (440 out of 502), making it useless for traceability.
- **No version tracking** for the model that produced each decision.
- **No audit log** connecting inputs to outputs.
- **No change history** showing whether any record was modified after the initial decision.

Without these logs, NovaCred cannot demonstrate compliance to regulators, investigate complaints, or trace how a specific decision was reached.